# 08 — Init → NeoBERT Load Integrity (no weight lost before training)

NeoBERT arch (from its `model.py`): input embeddings = `model.encoder` (`nn.Embedding`);
LM head = `NeoBERTLMHead.decoder` (untied `nn.Linear`); FFN uses SwiGLU (`w12`/`w3`) which
`salt3_common` patches when xformers is absent — a known silent-random-reinit hazard.

This notebook reloads a saved init and proves, BEFORE any GPU training is spent:
1. `from_pretrained` reports **zero** missing keys (nothing left at random init).
2. The reload's encoder/decoder/bias **byte-match** the raw safetensors in the folder.
3. No FFN/SwiGLU tensor is random (std-fingerprint check across all encoder params).
4. Forward pass on Vietnamese is finite; step-0 MLM loss is in the expected band.
5. The encoder weight equals the standalone `init_vietnamese_embeddings.pt`.

Set `INIT_NAME` to whichever artifact you want to certify.


In [1]:
%%capture
!pip install -U transformers safetensors huggingface_hub sentencepiece


In [2]:
import sys, math
from pathlib import Path
import numpy as np, torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
from salt3_common import extract_embedding_weight

INIT_NAME = 'videberta_salt_init_v5_globalmap_freqbias'   # artifact under test
INIT_DIR = PROJECT_ROOT / 'init' / INIT_NAME
MODEL_DIR = INIT_DIR / 'model'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('certifying:', MODEL_DIR, '| device', DEVICE)
checks = []   # (name, ok, detail)


Mounted at /content/drive
certifying: /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/model | device cuda


## 1. from_pretrained key audit — nothing left at random init

In [3]:
# Capture the missing/unexpected report by loading config + state_dict manually first.
from safetensors.torch import load_file
st_path = MODEL_DIR / 'model.safetensors'
ckpt = load_file(str(st_path)) if st_path.exists() else None
if ckpt is None:
    import glob
    print('no safetensors; files:', [p.name for p in MODEL_DIR.iterdir()])

model = AutoModelForMaskedLM.from_pretrained(MODEL_DIR, trust_remote_code=True).to(DEVICE).eval()
tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

model_keys = set(model.state_dict().keys())
ckpt_keys = set(ckpt.keys()) if ckpt else set()
missing = model_keys - ckpt_keys      # in model, NOT in file -> random init
unexpected = ckpt_keys - model_keys   # in file, NOT loaded
print(f'model tensors {len(model_keys)} | checkpoint tensors {len(ckpt_keys)}')
print(f'MISSING (random-init in model): {len(missing)}')
for k in sorted(missing)[:20]: print('   ', k)
print(f'UNEXPECTED (in file, unused): {len(unexpected)}')
for k in sorted(unexpected)[:20]: print('   ', k)
checks.append(('zero missing keys (no random reinit)', len(missing) == 0, f'{len(missing)} missing'))


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

model tensors 172 | checkpoint tensors 172
MISSING (random-init in model): 0
UNEXPECTED (in file, unused): 0


## 2. Encoder / decoder / bias byte-match the saved file

In [4]:
def match(name, model_t, key):
    if ckpt is None or key not in ckpt:
        checks.append((f'{name} present in file', False, f'key {key} absent'))
        print(f'  {name}: key {key} NOT in checkpoint'); return
    d = (model_t.detach().float().cpu() - ckpt[key].float()).abs().max().item()
    ok = d < 1e-4
    checks.append((f'{name} byte-match', ok, f'max|Δ|={d:.2e}'))
    print(f'  {name:18s} vs {key:28s} max|Δ| {d:.2e} {"OK" if ok else "MISMATCH"}')

enc = extract_embedding_weight(model)
match('encoder.weight', enc, 'model.encoder.weight')
match('decoder.weight', model.decoder.weight, 'decoder.weight')
match('decoder.bias', model.decoder.bias, 'decoder.bias')


  encoder.weight     vs model.encoder.weight         max|Δ| 0.00e+00 OK
  decoder.weight     vs decoder.weight               max|Δ| 0.00e+00 OK
  decoder.bias       vs decoder.bias                 max|Δ| 0.00e+00 OK


## 3. No FFN/SwiGLU tensor silently random (std fingerprint)

In [5]:
# Trained weights have varied std; a tensor at exactly init_range fingerprint is suspect.
rnd, tot, suspects = 0, 0, []
for name, p in model.named_parameters():
    if any(t in name for t in ('transformer_encoder', 'model.layers', 'w12', 'w3', 'ffn')):
        tot += 1
        s = p.detach().float().std().item()
        if s < 0.005 or abs(s - 0.02) < 0.001:   # near-zero or exactly uniform-init range
            rnd += 1; suspects.append((name, s))
print(f'encoder/FFN tensors checked: {tot}; suspicious(random-like): {rnd}')
for n, s in suspects[:12]: print(f'   {n}: std {s:.4f}')
checks.append(('FFN/SwiGLU not random', tot > 0 and rnd <= tot * 0.1, f'{rnd}/{tot} suspicious'))


encoder/FFN tensors checked: 168; suspicious(random-like): 4
   model.transformer_encoder.9.ffn_norm.weight: std 0.0200
   model.transformer_encoder.10.ffn_norm.weight: std 0.0197
   model.transformer_encoder.11.ffn_norm.weight: std 0.0195
   model.transformer_encoder.14.ffn_norm.weight: std 0.0196


## 4. Forward finite + step-0 MLM loss in band

In [6]:
sents = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng mạnh trong năm qua.',
         'Cô ấy đọc sách trong thư viện mỗi buổi chiều.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']
enc_in = tok(sents, padding=True, truncation=True, max_length=48, return_tensors='pt').to(DEVICE)
ids = enc_in['input_ids']
torch.manual_seed(0)
special = set(tok.all_special_ids)
pm = torch.full(ids.shape, 0.20)
for sid in special: pm[ids.cpu() == sid] = 0.0
masked = torch.bernoulli(pm).bool().to(DEVICE)
labels = torch.full_like(ids, -100); labels[masked] = ids[masked]
mids = ids.clone(); mids[masked] = tok.mask_token_id
with torch.no_grad():
    logits = model(input_ids=mids, attention_mask=enc_in['attention_mask']).logits
fin = torch.isfinite(logits).all().item()
loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100).item()
V = model.config.vocab_size
print(f'forward finite: {fin} | step-0 MLM loss {loss:.3f} (random {math.log(V):.3f})')
checks.append(('forward finite', fin, ''))
checks.append(('step-0 loss < random', fin and loss < math.log(V), f'{loss:.2f} vs {math.log(V):.2f}'))


forward finite: False | step-0 MLM loss nan (random 10.326)


## 5. Encoder == standalone init tensor + verdict

In [7]:
pt = INIT_DIR / 'init_vietnamese_embeddings.pt'
if pt.exists():
    saved = torch.load(pt, map_location='cpu').float()
    d = (enc.detach().float().cpu() - saved).abs().max().item()
    checks.append(('encoder == init_vietnamese_embeddings.pt', d < 1e-4, f'max|Δ|={d:.2e}'))
    print(f'encoder vs init_vietnamese_embeddings.pt: max|Δ| {d:.2e}')
else:
    print('init_vietnamese_embeddings.pt not found (skipped)')

print('\n' + '=' * 64); print(f'LOAD INTEGRITY — {INIT_NAME}'); print('=' * 64)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:42s} {detail}")
allok = all(ok for _, ok, _ in checks)
print('=' * 64)
print('✅ INIT LOADS FULLY INTO NEOBERT — safe to train' if allok
      else '❌ LOAD DEFECT — fix before spending GPU (see FAIL rows)')
print('=' * 64)


encoder vs init_vietnamese_embeddings.pt: max|Δ| 0.00e+00

LOAD INTEGRITY — videberta_salt_init_v5_globalmap_freqbias
  [PASS] zero missing keys (no random reinit)       0 missing
  [PASS] encoder.weight byte-match                  max|Δ|=0.00e+00
  [PASS] decoder.weight byte-match                  max|Δ|=0.00e+00
  [PASS] decoder.bias byte-match                    max|Δ|=0.00e+00
  [PASS] FFN/SwiGLU not random                      4/168 suspicious
  [FAIL] forward finite                             
  [FAIL] step-0 loss < random                       nan vs 10.33
  [PASS] encoder == init_vietnamese_embeddings.pt   max|Δ|=0.00e+00
❌ LOAD DEFECT — fix before spending GPU (see FAIL rows)
